In [78]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
alert = Alert('1h','GGAL')

In [80]:
pre_df = yf.download(tickers='GGAL', period='200d', interval='1h')
pre_df = pre_df.iloc[:-3 , :]
display(pre_df)

C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1794724637.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  pre_df = yf.download(tickers='GGAL', period='200d', interval='1h')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GGAL,GGAL,GGAL,GGAL,GGAL
Datetime,,,,,
2024-09-03 13:30:00+00:00,39.139999,39.810001,38.700100,39.680000,278031
2024-09-03 14:30:00+00:00,39.680000,39.929901,38.720001,39.150002,138622
2024-09-03 15:30:00+00:00,39.970001,40.040001,39.520802,39.680000,133388
2024-09-03 16:30:00+00:00,41.060001,41.230000,39.950001,39.989498,302079
2024-09-03 17:30:00+00:00,41.025002,41.110001,40.709999,41.044998,215399
...,...,...,...,...,...
2025-06-18 19:30:00+00:00,54.090000,54.230000,54.005001,54.025002,102481
2025-06-20 13:30:00+00:00,52.590000,54.080002,52.230000,54.080002,109579


pre_df = alert.set_test(pre_df, "1h")

In [94]:
# Stochastic calculation
def stochastic(df, i, K, D, smoth):
        
    df["k"] = (100. * (df.Close - df.Low.rolling(K).min()) /
        (df.High.rolling(K).max() - df.Low.rolling(K).min()))
    
    df["k" + i] = df.k.rolling(smoth).mean()
    df["d" + i] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  

    return df

In [102]:
def set_test( df, period):
        k = 17   
        d = 5    
        smth = 8   
        dayM = 5    
        semM = 4

        # Convert time
        df["time"] = pd.to_datetime(df.index, utc=True)
        df['timeArg'] = df['time'].dt.tz_convert('America/Argentina/Buenos_Aires')
        df['time'] = df['time'].dt.tz_convert(None)

        df = stochastic(df, "hora", k, d, smth)
        df = stochastic(df, "dia", k*dayM, d*dayM, smth*dayM)
        df = stochastic(df, "sem", k*dayM*semM, d*dayM*semM, smth*dayM*semM)
        #df = set_signals(df)
    
        return df
df = set_test(pre_df, "1h")
display(df )

C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1787476449.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)
C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1787476449.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)
C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1787476449.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)


Price,Close,High,Low,Open,Volume,time,timeArg,khora,dhora,kdia,ddia,ksem,dsem
Ticker,GGAL,GGAL,GGAL,GGAL,GGAL,,,,,,,,
Datetime,,,,,,,,,,,,,
2024-09-03 13:30:00+00:00,39.139999,39.810001,38.700100,39.680000,278031,2024-09-03 13:30:00,2024-09-03 10:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-03 14:30:00+00:00,39.680000,39.929901,38.720001,39.150002,138622,2024-09-03 14:30:00,2024-09-03 11:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-03 15:30:00+00:00,39.970001,40.040001,39.520802,39.680000,133388,2024-09-03 15:30:00,2024-09-03 12:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-03 16:30:00+00:00,41.060001,41.230000,39.950001,39.989498,302079,2024-09-03 16:30:00,2024-09-03 13:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
2024-09-03 17:30:00+00:00,41.025002,41.110001,40.709999,41.044998,215399,2024-09-03 17:30:00,2024-09-03 14:30:00-03:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-18 19:30:00+00:00,54.090000,54.230000,54.005001,54.025002,102481,2025-06-18 19:30:00,2025-06-18 16:30:00-03:00,66.078642,64.565738,27.684889,25.613147,64.967160,71.939414
2025-06-20 13:30:00+00:00,52.590000,54.080002,52.230000,54.080002,109579,2025-06-20 13:30:00,2025-06-20 10:30:00-03:00,60.565641,64.546386,27.469822,25.799733,64.658034,71.812308


In [121]:
# Seleciona as linhas que possuem pelo menos um valor NaN
rows_with_nan = df[df.isna().any(axis=1)]
display(rows_with_nan)

# Remove as linhas que possuem valores NaN e salva o resultado em um novo DataFrame (ou atualize o mesmo)
df_clean = df.dropna()

# Se preferir modificar o DataFrame original, use inplace=True:
# df.dropna(inplace=True)

Price,Close,High,Low,Open,Volume,time,timeArg,khora,dhora,kdia,...,compra,venta,closeLong,closeShort,entradaLongH,entradaShortH,oper,long,short,state
Ticker,GGAL,GGAL,GGAL,GGAL,GGAL,,,,,,...,,,,,,,,,,
Datetime,,,,,,,,,,,,,,,,,,,,,
2024-09-03 13:30:00+00:00,39.139999,39.810001,38.700100,39.680000,278031,2024-09-03 13:30:00,2024-09-03 10:30:00-03:00,NaN,NaN,NaN,...,False,False,False,False,False,False,,,,
2024-09-03 14:30:00+00:00,39.680000,39.929901,38.720001,39.150002,138622,2024-09-03 14:30:00,2024-09-03 11:30:00-03:00,NaN,NaN,NaN,...,False,False,False,False,False,False,,,,neutro
2024-09-03 15:30:00+00:00,39.970001,40.040001,39.520802,39.680000,133388,2024-09-03 15:30:00,2024-09-03 12:30:00-03:00,NaN,NaN,NaN,...,False,False,False,False,False,False,,,,neutro
2024-09-03 16:30:00+00:00,41.060001,41.230000,39.950001,39.989498,302079,2024-09-03 16:30:00,2024-09-03 13:30:00-03:00,NaN,NaN,NaN,...,False,False,False,False,False,False,,,,neutro
2024-09-03 17:30:00+00:00,41.025002,41.110001,40.709999,41.044998,215399,2024-09-03 17:30:00,2024-09-03 14:30:00-03:00,NaN,NaN,NaN,...,False,False,False,False,False,False,,,,neutro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-03 19:30:00+00:00,69.410004,69.820000,69.220001,69.779999,114954,2025-01-03 19:30:00,2025-01-03 16:30:00-03:00,81.556024,85.385461,57.091965,...,False,False,False,False,False,False,,,,neutro
2025-01-03 20:30:00+00:00,70.160004,70.190002,69.349998,69.349998,207743,2025-01-03 20:30:00,2025-01-03 17:30:00-03:00,80.908343,83.688590,57.879872,...,False,False,False,False,False,False,,,,neutro


In [104]:

def set_compra( df, i):
        df.loc[i, "oper"] = "COMPRA"


def set_venta( df, i):
        df.loc[i, "oper"] = "VENTA"


def close_long( df, i, compra, venta):
        df.loc[i, "oper"] = "CLOSELONG"
        if compra !=0.:
            df.loc[i, "long"] = (venta - compra) / compra


def close_short( df, i, compra, venta):
        df.loc[i, "oper"] = "CLOSESHORT"
        if venta !=0.:
            df.loc[i, "short"] = (venta - compra) / venta

In [123]:
def set_signals(df):
        state = ""
        df["compra"] = False
        df["venta"] = False
        df["closeLong"] = False
        df["closeShort"] = False
        df["entradaLongH"] = False
        df["entradaShortH"] = False
        df["oper"] = ""
        df["long"] = ""
        df["short"] = ""
        df["state"] = "neutro"

        compra = 0.
        venta = 0.
    
    #alert.py part 2
       
        
        lastEntradaLongH = False
        lastEntradaShortH = False
        
        #for i in df.index:
        for i, row in df.iterrows():            
            l = df.loc[i]
            df.loc[i, "state"] = state

            # ESTRATEGIA 1 - Stochastico alineado y esperar cruce
            if (l.khora > 20. and l.khora > l.dhora):
                df.loc[i, "entradaLongH"] = True
                if (not lastEntradaLongH):
                    if (l.kdia > 20. and l.kdia > l.ddia):
                        if(state == "venta"):
                            df.loc[i, "closeShort"] = True
                            state = "neutro"
                            compra = l.Close
                            close_short(df, i, compra, venta)
                        if (l.ksem > 80. or l.ksem > l.dsem):
                            df.loc[i, "compra"] = True
                            if state != "compra":
                                state = "compra"
                                compra = l.Close
                                set_compra(df, i)
            else:
                df.loc[i, "entradaLongH"] = False

            if (l.khora < 80. and l.khora < l.dhora):
                df.loc[i, "entradaShortH"] = True
                if (not lastEntradaShortH):
                    if(l.kdia < 80. and l.kdia < l.ddia):
                        if(state == "compra"):
                            df.loc[i, "closeLong"] = True
                            state = "neutro"
                            venta = l.Close
                            close_long(df, i, compra, venta)
                        if(l.ksem < 20. or l.ksem < l.dsem):
                            df.loc[i, "venta"] = True
                            if state != "venta":
                                state = "venta"
                                venta = l.Close
                                set_venta(df, i)
            else:
                df.loc[i, "entradaShortH"] = False

            lastEntradaLongH = df.loc[i, "entradaLongH"]
            lastEntradaShortH = df.loc[i, "entradaShortH"]
    
        return df
display(set_signals(df_clean))

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [51]:
df = stochastic(pre_df,"1h",17,5,8)
last = df.iloc[-1]
display (last)

# Read last alert csv
lastAlert = alert.read_alert_csv('lastAlert_1h.csv')
display(lastAlert)

C:\Users\scitr\AppData\Local\Temp\ipykernel_7528\1567423116.py:10: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df.drop(columns=["k"], inplace=True)


Price   Ticker
Close   GGAL         55.020000
High    GGAL         55.299999
Low     GGAL         54.919998
Open    GGAL         55.090000
Volume  GGAL      80081.000000
k1h                  63.429191
d1h                  51.514441
Name: 2025-06-18 16:30:00+00:00, dtype: float64

Unnamed: 0
Open                      8.69849967956543
High                     8.930000305175781
Low                      8.680000305175781
Close                    8.869999885559082
Adj Close                8.869999885559082
Volume                              147519
time                   2022-12-21 16:30:00
timeArg          2022-12-21 13:30:00-03:00
khora                    89.27899126024955
dhora                     89.2503455622804
kdia                    51.219426452147296
ddia                     41.25813962855897
ksem                     44.75414109800853
dsem                     32.42956752293771
compra                                True
venta                                False
closeLong                            False
closeShort                           False
entradaLongH                          True
entradaShortH                        False
oper                                COMPRA
long                                   NaN
short                                  NaN
